# J3: Joint TiDE MTL
Train the joint model (load+price+aux heads, load->price pathway) and run the per-target STL-vs-MTL gate: ship a head only if the shared trunk does not regress its val pinball vs a single-task DL model.

In [ ]:
import sys; sys.path.insert(0, '../../src')
import os; os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import pandas as pd, numpy as np, torch
from data.loaders import load_split
from module_joint.config import JointConfig, select_device
LQ = pd.read_parquet('../../data/module_a/load_quantiles.parquet')
device = select_device(); print('device', device)


In [ ]:
from module_joint.train import train_one
from module_joint.evaluate import compare_price, compare_load, make_truth
train, val = load_split('train'), load_split('val')
val_ctx = pd.concat([train.tail(168), val])
cfg = JointConfig(backbone='tide', max_epochs=80)
est, hist = train_one(cfg, train, val, LQ, device=device)
preds = est.predict_quantiles(val_ctx, LQ, restrict_to=val.index)
for tgt in ['load','price']:
    tr = make_truth(val_ctx, preds[tgt].index, tgt)
    fn = compare_price if tgt=='price' else compare_load
    print(tgt, fn(preds[tgt], tr)['overall'])
